# ✈️ NASA Turbofan Engine RUL Prediction — PyTorch Only

Predicting **Remaining Useful Life (RUL)**: given a window of recent sensor readings from a jet engine, how many more cycles until it needs maintenance? This is a rewrite of `NASA.ipynb` using the [NASA C-MAPSS](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/) turbofan degradation dataset (FD001 subset: 100 engines, run to failure).


- **PyTorch only** for every model — no other deep learning framework
- **pandas / numpy / matplotlib** for data handling and plotting
- **scikit-learn used only for the benchmark model, data splitting, and scoring metrics** — not for any of the deep learning itself. Splitting uses `GroupShuffleSplit` (the standard tool for "split by group, not by row"); scaling is still done by hand, which is simple enough not to need a library
- **4 PyTorch models**, trained and compared with one shared, reusable training function
- Every step and function gets a short explanation — this is written to be skimmed and referenced later, not just run once

### The pipeline
```
1. Download & extract the dataset
2. Load train/test/RUL files, quick EDA
3. Drop constant sensors, engineer the RUL label (with the standard "piecewise-linear" cap)
4. Split by engine into train/validation (never by row — that would leak time steps from the same engine)
5. Standardize features using train-set statistics only
6. Slide a fixed-size window over each engine's history -> (X, y) sequences
7. Train 4 PyTorch models + 1 scikit-learn benchmark with one shared training loop
8. Evaluate all 5 on the official, untouched test engines
9. Compare
```


---
## 1. Download & extract the dataset

The file NASA hosts is a zip **inside** a zip (a folder gets zipped along with the data), so it takes two extraction passes to get to the actual `.txt` files.

| Function | What it does |
|---|---|
| `requests.get(url)` | downloads the file's bytes over HTTP |
| `response.raise_for_status()` | raises an error immediately if the download failed (bad URL, server error, etc.) instead of silently saving a broken file |
| `zipfile.ZipFile(path).extractall(dir)` | unpacks a `.zip` archive into a folder |


In [ ]:
import requests

url = "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip"

response = requests.get(url)
response.raise_for_status()

with open("/content/CMAPSSData.zip", "wb") as f:
    f.write(response.content)

print("Downloaded.")

In [ ]:
import zipfile

# First pass: unzip the outer archive
with zipfile.ZipFile("/content/CMAPSSData.zip", "r") as zip_ref:
    zip_ref.extractall("/content/CMaps")

# Second pass: the outer archive contains another zip with the actual data files
inner_zip = "/content/CMaps/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData.zip"
with zipfile.ZipFile(inner_zip, "r") as zip_ref:
    zip_ref.extractall("/content/CMaps/CMAPSSData")

print("Extracted.")

## 2. Imports

| Import | Purpose |
|---|---|
| `torch`, `nn`, `torch.utils.data` | tensors, model building blocks, `Dataset`/`DataLoader` — the only deep learning framework used |
| `sklearn.ensemble.RandomForestRegressor` | the **one** non-PyTorch *model*, used purely as a benchmark to compare against |
| `sklearn.model_selection.GroupShuffleSplit` | splits data by group (by engine) rather than by row — used for the train/validation split |
| `sklearn.metrics` | scoring functions (MAE, RMSE, R²) — used to score every model, PyTorch or not, on equal footing |
| `pandas`, `numpy`, `matplotlib.pyplot` | data handling and plotting |


In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| Device:", device)

---
## 3. Load the data

| Function | What it does |
|---|---|
| `os.walk(dir)` | recursively lists every folder/file under `dir` — used here to *find* a file without knowing its exact nested path in advance |
| `pd.read_csv(path, sep=r"\s+", header=None, names=[...])` | C-MAPSS files are whitespace-separated with no header row, so both are specified explicitly |


In [ ]:
def find_file(filename, base="/content/CMaps"):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"{filename} not found under {base}")

train_path = find_file("train_FD001.txt")
test_path = find_file("test_FD001.txt")
rul_path = find_file("RUL_FD001.txt")

print("Train:", train_path)
print("Test :", test_path)
print("RUL  :", rul_path)

In [ ]:
columns = (
    ["unit_number", "time_in_cycles", "operational_setting_1", "operational_setting_2", "operational_setting_3"]
    + [f"sensor_measurement_{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(train_path, sep=r"\s+", header=None, names=columns)
test_df = pd.read_csv(test_path, sep=r"\s+", header=None, names=columns)
rul_df = pd.read_csv(rul_path, sep=r"\s+", header=None, names=["RUL"])

print("train_df:", train_df.shape, "| test_df:", test_df.shape, "| rul_df:", rul_df.shape)
train_df.head()

## 4. Quick EDA: missing values & constant sensors


| Function | What it does |
|---|---|
| `DataFrame.nunique()` | count of distinct values per column — `1` means the column never changes (constant) |


In [ ]:
print("Missing values:", train_df.isna().sum().sum())
print("Training engines:", train_df["unit_number"].nunique())
print("Test engines:", test_df["unit_number"].nunique())

sensor_cols = [f"sensor_measurement_{i}" for i in range(1, 22)]
constant_sensors = train_df[sensor_cols].columns[train_df[sensor_cols].nunique() <= 1].tolist()
print("\nConstant sensors (to be dropped):", constant_sensors)

setting_cols = ["operational_setting_1", "operational_setting_2", "operational_setting_3"]
feature_cols = [c for c in setting_cols + sensor_cols if c not in constant_sensors]
print("Features kept:", len(feature_cols), "/", len(setting_cols + sensor_cols))

## 5. Visualize sensor behavior

A correlation heatmap across sensors, and one engine's raw trend for a few sensors — useful for building intuition for *why* this is learnable: some sensors clearly drift as the engine degrades.


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(train_df[feature_cols].corr(), aspect="auto", cmap="coolwarm")
plt.colorbar()
plt.xticks(range(len(feature_cols)), feature_cols, rotation=90, fontsize=7)
plt.yticks(range(len(feature_cols)), feature_cols, fontsize=7)
plt.title("Feature Correlation (train set)")
plt.tight_layout()
plt.show()

In [ ]:
engine_1 = train_df[train_df["unit_number"] == 1]

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, sensor in zip(axs, ["sensor_measurement_2", "sensor_measurement_3", "sensor_measurement_4"]):
    ax.plot(engine_1["time_in_cycles"], engine_1[sensor])
    ax.set_xlabel("Cycle")
    ax.set_title(sensor)
plt.tight_layout()
plt.show()

---
## 6. Engineer the RUL label

Each training engine runs until failure, so at every cycle: `RUL = (final cycle of that engine) - (current cycle)`.

| Function | What it does |
|---|---|
| `groupby(col)[other_col].max()` | the maximum value of `other_col` within each group |
| `Series.clip(upper=x)` | caps values at `x`, leaving smaller values unchanged |


In [ ]:
RUL_CAP = 125

max_cycle = train_df.groupby("unit_number")["time_in_cycles"].max().rename("max_cycle")
train_df = train_df.merge(max_cycle, on="unit_number", how="left")
train_df["RUL"] = (train_df["max_cycle"] - train_df["time_in_cycles"]).clip(upper=RUL_CAP)
train_df.drop(columns=["max_cycle"], inplace=True)

engine_1 = train_df[train_df["unit_number"] == 1]  # re-filter now that RUL has been added
plt.figure(figsize=(10, 4))
plt.plot(engine_1["time_in_cycles"], engine_1["RUL"])
plt.xlabel("Cycle"); plt.ylabel("RUL (capped)"); plt.title("Engine 1 — piecewise-linear RUL target")
plt.show()

---
## 7. Train / validation split — by engine, not by row

Splitting individual rows randomly would put consecutive cycles from the *same* engine's life into both train and validation — the model would essentially be tested on data adjacent to what it trained on. `GroupShuffleSplit` is scikit-learn's tool for exactly this: it shuffles and splits by *group* (here, `unit_number`), guaranteeing every engine's rows all end up on the same side of the split.

| Function | What it does |
|---|---|
| `GroupShuffleSplit(n_splits, test_size, random_state)` | a splitter object configured for one group-aware train/test split |
| `splitter.split(X, groups=...)` | returns `(train_idx, val_idx)` — row positions for each side, computed so that no group straddles both |


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(train_df, groups=train_df["unit_number"]))

train_split = train_df.iloc[train_idx].copy()
val_split = train_df.iloc[val_idx].copy()

print(f"Train engines: {train_split['unit_number'].nunique()} ({len(train_split)} rows)")
print(f"Val engines:   {val_split['unit_number'].nunique()} ({len(val_split)} rows)")

## 8. Standardize features

Neural networks train better when every feature is on a similar scale. Mean and standard deviation are computed **only from the training split** and then applied to validation and test — fitting on data the model will later be evaluated on would leak information about that data's distribution.


In [ ]:
train_mean = train_split[feature_cols].mean()
train_std = train_split[feature_cols].std()

train_split[feature_cols] = (train_split[feature_cols] - train_mean) / train_std
val_split[feature_cols] = (val_split[feature_cols] - train_mean) / train_std

train_split[feature_cols].describe().T[["mean", "std"]].head()

---
## 9. Slide a window over each engine's history

A neural net needs to see a short *history* of readings to spot a trend, not just one cycle in isolation. `WINDOW_SIZE=30` consecutive cycles become one input sequence; the label is the RUL at the last cycle of that window. This is applied independently per engine (and each engine is sorted chronologically first) so no window ever mixes cycles from two different engines.

| Function | What it does |
|---|---|
| `df.groupby(col)` | iterate over each group (each engine) separately |
| `array[i : i + w]` | a length-`w` slice starting at position `i` — the sliding window |


In [ ]:
WINDOW_SIZE = 30

def create_sequences(df, feature_cols, window_size):
    X, y = [], []
    for _, engine in df.groupby("unit_number"):
        engine = engine.sort_values("time_in_cycles")
        features, rul = engine[feature_cols].values, engine["RUL"].values

        for i in range(len(engine) - window_size + 1):
            X.append(features[i : i + window_size])
            y.append(rul[i + window_size - 1])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


X_train, y_train = create_sequences(train_split, feature_cols, WINDOW_SIZE)
X_val, y_val = create_sequences(val_split, feature_cols, WINDOW_SIZE)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("NaNs in X_train:", np.isnan(X_train).sum())

## 10. `Dataset` & `DataLoader`


| Class | What it does |
|---|---|
| `Dataset` | wraps `X`/`y` arrays as tensors and defines `__len__`/`__getitem__`, the two methods `DataLoader` needs |
| `DataLoader(dataset, batch_size, shuffle)` | serves the `Dataset` in mini-batches |


In [ ]:
class RULDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64
train_loader = DataLoader(RULDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(RULDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))

---
## 11. Prepare the official test set

| Function | What it does |
|---|---|
| `np.repeat(row, n, axis=0)` | repeats a row `n` times, stacked as new rows |
| `np.vstack([a, b])` | stacks arrays on top of each other |


In [ ]:
test_df[feature_cols] = (test_df[feature_cols] - train_mean) / train_std

def create_test_sequences(df, feature_cols, window_size):
    X = []
    for _, engine in df.groupby("unit_number"):  # groupby sorts by unit_number ascending, matching rul_df's row order
        engine = engine.sort_values("time_in_cycles")
        features = engine[feature_cols].values

        if len(features) < window_size:
            pad = np.repeat(features[0:1], window_size - len(features), axis=0)
            features = np.vstack([pad, features])

        X.append(features[-window_size:])  # only the most recent window matters for the final prediction

    return np.array(X, dtype=np.float32)


X_test = create_test_sequences(test_df, feature_cols, WINDOW_SIZE)
y_test = rul_df["RUL"].values.clip(max=RUL_CAP).astype(np.float32)  # same cap as training, for a fair comparison

print("X_test:", X_test.shape, "y_test:", y_test.shape)

---
## 12. scikit-learn benchmark

The **only** non-PyTorch model here — a `RandomForestRegressor` on the flattened windows, purely to answer "is the extra complexity of a neural net actually earning its keep?"

| Function | What it does |
|---|---|
| `array.reshape(n, -1)` | flattens each `(window_size, n_features)` sequence into one long vector, since Random Forest doesn't understand sequence structure |
| `RandomForestRegressor(n_estimators, max_depth)` | an ensemble of decision trees; `n_estimators` = how many trees, `max_depth` caps each tree's depth to control overfitting/runtime |


In [ ]:
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train_flat, y_train)

rf_val_mae = mean_absolute_error(y_val, rf_model.predict(X_val_flat))
print(f"Random Forest benchmark — Validation MAE: {rf_val_mae:.2f} cycles")

---
## 13. Four PyTorch models

Every model takes a `(batch, WINDOW_SIZE, n_features)` sequence and outputs one number (predicted RUL) per sequence — same input, same output, four different ways of processing what's in between.

| Class | What it does |
|---|---|
| `nn.Flatten()` | collapses the whole window into one long vector, ignoring time order |
| `nn.Conv1d(in_channels, out_channels, kernel_size, padding)` | slides learnable filters along the *time* axis, picking up local trends a few cycles wide |
| `nn.AdaptiveAvgPool1d(1)` | averages a sequence down to one value per channel, regardless of window length |
| `nn.LSTM(input_size, hidden_size, batch_first=True)` | reads the sequence step by step, keeping a memory cell that can retain information over long ranges |
| `nn.GRU(input_size, hidden_size, batch_first=True)` | a recurrent unit like LSTM but with a simpler gating mechanism — fewer parameters, often trains faster, similar accuracy |


In [ ]:
N_FEATURES = len(feature_cols)


class MLPModel(nn.Module):
    """Flattens the window and applies fully-connected layers -- ignores time order entirely, the simplest option."""
    def __init__(self, window_size, n_features, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(window_size * n_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class CNN1DModel(nn.Module):
    """1D convolutions slide over the time axis, learning short local degradation patterns."""
    def __init__(self, n_features, hidden=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, hidden, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)              # (batch, window, features) -> (batch, features, window)
        x = self.conv(x).squeeze(-1)
        return self.head(x).squeeze(1)


class LSTMModel(nn.Module):
    """Reads the window one cycle at a time, keeping a running memory -- the classic choice for RUL prediction."""
    def __init__(self, n_features, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)           # h_n: final hidden state, shape (1, batch, hidden)
        return self.head(h_n[-1]).squeeze(1)


class GRUModel(nn.Module):
    """Same idea as LSTM, simpler internal gating -- fewer parameters, usually trains a bit faster."""
    def __init__(self, n_features, hidden=64):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        _, h_n = self.gru(x)
        return self.head(h_n[-1]).squeeze(1)


# name -> a zero-arg function that builds a fresh instance
model_builders = {
    "MLP":     lambda: MLPModel(WINDOW_SIZE, N_FEATURES),
    "1D-CNN":  lambda: CNN1DModel(N_FEATURES),
    "LSTM":    lambda: LSTMModel(N_FEATURES),
    "GRU":     lambda: GRUModel(N_FEATURES),
}

---
## 14. One reusable training loop for every model

Same idea as picking any model above: one `train_model()` function, reused for all four, instead of four copy-pasted training loops.

| Function | What it does |
|---|---|
| `nn.MSELoss()` | Mean Squared Error — the standard loss for regression; penalizes large errors more than small ones |
| `mean_absolute_error(y_true, y_pred)` | from `sklearn.metrics` — average `\|prediction - actual\|`, in the same units as RUL (cycles), easier to read than MSE |


In [ ]:
EPOCHS = 10
LR = 1e-3


def train_model(model, name, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    model.to(device)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_mae": []}
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            preds = model(X_batch)
            loss = loss_fn(preds, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        val_loss, val_preds, val_true = 0, [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = model(X_batch)
                val_loss += loss_fn(preds, y_batch).item()
                val_preds.extend(preds.cpu().tolist())
                val_true.extend(y_batch.cpu().tolist())

        val_mae = mean_absolute_error(val_true, val_preds)
        history["train_loss"].append(total_loss / len(train_loader))
        history["val_loss"].append(val_loss / len(val_loader))
        history["val_mae"].append(val_mae)

        print(f"[{name}] Epoch {epoch+1}/{epochs} | Train MSE {history['train_loss'][-1]:.1f} | Val MAE {val_mae:.2f} cycles")

    return history, time.time() - start_time


def evaluate(model, X_test, y_test):
    """Runs a trained PyTorch model once on the held-out test sequences and scores it against the true RUL."""
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()
    return {
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": mean_squared_error(y_test, preds) ** 0.5,
        "R2": r2_score(y_test, preds),
    }

---
## 15. Train & evaluate all 4 PyTorch models


In [ ]:
results = {}

for name, build_fn in model_builders.items():
    print("=" * 60)
    model = build_fn()
    history, elapsed = train_model(model, name, train_loader, val_loader)
    test_scores = evaluate(model, X_test, y_test)

    results[name] = {"history": history, "train_time_sec": elapsed, **test_scores}
    print(f"-> {name}: Test MAE={test_scores['MAE']:.2f} | RMSE={test_scores['RMSE']:.2f} | R2={test_scores['R2']:.3f}")

---
## 16. Compare all 5 (4 PyTorch models + the Random Forest benchmark)

| Function | What it does |
|---|---|
| `pd.DataFrame(dict).T` | dict-of-dicts → DataFrame, transposed so each outer key becomes a row |


In [ ]:
rf_test_preds = rf_model.predict(X_test_flat)
results["Random Forest (benchmark)"] = {
    "MAE": mean_absolute_error(y_test, rf_test_preds),
    "RMSE": mean_squared_error(y_test, rf_test_preds) ** 0.5,
    "R2": r2_score(y_test, rf_test_preds),
    "train_time_sec": float("nan"),  # not measured the same way as the PyTorch loop
}

comparison_df = pd.DataFrame({
    name: {"Test MAE (cycles)": round(r["MAE"], 2), "Test RMSE": round(r["RMSE"], 2), "Test R2": round(r["R2"], 3)}
    for name, r in results.items()
}).T.sort_values("Test MAE (cycles)")

comparison_df

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 5))

names = list(results.keys())
maes = [results[n]["MAE"] for n in names]
axs[0].bar(names, maes, color="darkorange")
axs[0].set_title("Test MAE by Model (lower is better)")
axs[0].set_ylabel("MAE (cycles)")
axs[0].tick_params(axis="x", rotation=30)

for name in model_builders:  # only the PyTorch models have a training curve
    axs[1].plot(results[name]["history"]["val_mae"], label=name)
axs[1].set_title("Validation MAE over Epochs")
axs[1].set_xlabel("Epoch")
axs[1].set_ylabel("MAE (cycles)")
axs[1].legend()

plt.tight_layout()
plt.show()

A lower MAE means predictions are, on average, closer to the true remaining cycles. The Random Forest benchmark is a useful sanity check: if a PyTorch model can't beat it, the extra training time and complexity of a neural net isn't buying you anything on this dataset. Among the PyTorch models, watch whether the recurrent models (LSTM/GRU) actually outperform the simpler MLP/CNN — sequence order matters more for some degradation patterns than others.


---
# 📋 Cheat Sheet — every function/class used in this notebook

### Download & data loading
| Function | What it does |
|---|---|
| `requests.get(url)` / `.raise_for_status()` | download a file; fail loudly if it didn't work |
| `zipfile.ZipFile(path).extractall(dir)` | unpack a `.zip` |
| `os.walk(dir)` | recursively list files/folders |
| `pd.read_csv(path, sep, header, names)` | load a delimited text file into a DataFrame |

### Feature engineering & splitting
| Function | What it does |
|---|---|
| `DataFrame.nunique()` | distinct-value count per column (used to find constant columns) |
| `groupby(col)[other].max()` | max of `other` within each group |
| `Series.clip(upper=x)` | cap values at `x` |
| `GroupShuffleSplit(n_splits, test_size, random_state)` | configures a group-aware train/test split |
| `splitter.split(X, groups=...)` | returns row indices for each side, keeping every group intact |
| `(x - mean) / std` | manual standardization — same math `StandardScaler` does internally |

### Building sequences
| Function | What it does |
|---|---|
| `df.groupby(col)` | iterate per group (per engine) |
| `array[i:i+w]` | fixed-length slice — the sliding window |
| `np.repeat(row, n, axis=0)` / `np.vstack` | pad a short sequence up to a fixed length |

### PyTorch data pipeline
| Class | What it does |
|---|---|
| `Dataset` (`__len__`, `__getitem__`) | required interface for any custom dataset |
| `DataLoader(dataset, batch_size, shuffle)` | batches a `Dataset` |
| `torch.tensor(x, dtype=...)` | NumPy array → tensor |

### PyTorch models
| Class | What it does |
|---|---|
| `nn.Flatten()` | collapses all non-batch dimensions into one |
| `nn.Linear(in, out)` | fully-connected layer |
| `nn.Conv1d(in_ch, out_ch, kernel_size, padding)` | 1-D convolution — learnable filters sliding along a sequence |
| `nn.AdaptiveAvgPool1d(output_size)` | pools a sequence to a fixed length regardless of input length |
| `nn.LSTM(input_size, hidden_size, batch_first=True)` | recurrent layer with a long-term memory cell |
| `nn.GRU(input_size, hidden_size, batch_first=True)` | recurrent layer, simpler gating than LSTM |
| `tensor.permute(dims)` | reorders a tensor's dimensions (used to match `Conv1d`'s expected `(batch, channels, length)`) |

### Training & evaluation
| Function | What it does |
|---|---|
| `nn.MSELoss()` | Mean Squared Error — standard regression loss |
| `torch.optim.Adam(params, lr)` | adaptive optimizer |
| `optimizer.zero_grad()` / `loss.backward()` / `optimizer.step()` | the standard gradient update sequence |
| `model.train()` / `model.eval()` / `torch.no_grad()` | mode switches + disabling gradient tracking for evaluation |
| `mean_absolute_error`, `mean_squared_error`, `r2_score` | from `sklearn.metrics` — used to score every model uniformly |
| `RandomForestRegressor(n_estimators, max_depth)` | the one non-PyTorch model, used only as a benchmark |
| `time.time()` | wall-clock timing |

### Comparing results
| Function | What it does |
|---|---|
| `pd.DataFrame(dict).T` | dict-of-dicts → DataFrame, rows = outer keys |
| `DataFrame.sort_values(col)` | sort rows by a column |
| `plt.subplots`, `ax.bar`, `ax.plot` | plotting |

---

